# Does more training data keep helping? d0256 x 80000 on the shared pool

**Runs:** Slurm `48912548` (d0256 x n_train=80000, 300 epochs, 5:33:11),
`48935109` (d0128 x n_train=20000 + probes + eval, 1:07:26),
`48999377` (all seven checkpoints re-scored on the shared pool + probes on the
80000-instance model).

**Git commit:** `dc3598b` (branch `sudoku-dd`). Scripts
`scripts/slurm/sudoku2_d256_data16x.sbatch`,
`scripts/slurm/sudoku2_d128_data4x_probe.sbatch`,
`scripts/slurm/sudoku2_shared_pool_full_grid.sbatch`.

**Config** (one variable changed per run; everything else held):
`nhead=4, num_layers=12, dim_feedforward=1024, lr=1e-3, norm_first=True,
n_nodes=24, degree=5, n_colors=4, rho in [0.3, 0.95], hint_mode=removal,
scratchpad=False, n_val_instances=300, batch_size=32, epochs=300`.

**Seeds.** Training runs are single-seed (`sudoku_hparam_sweep` defaults). The
shared eval pool is `seed_start=50_000_000_000`, 1500 instances, disjoint from
every training set (max training seed 80000) and from every other eval module's
base. Probes use seeds 0/1/2, which shift both instance pools
(`500_000_000+seed` train, `900_000_000+seed` test) and the position sampling;
the spread across those three is the measurement noise quoted below.

## Hypothesis (stated before the eval ran)

`48912548`'s own-validation `agg_solve_rate` came back **0.9933**, up from
0.9733 at `n_train=20000`. I predicted the shared pool would put it at or above
the current best of 0.9847, with a positive paired delta, because 4x the data
cut the train/val gap from +0.817 to +0.095. The registered falsifier was a
negative delta, which would mean the collapse in its easy-end metrics
(`solve_rate_k0` 0.60 -> 0.07, `mean_o1` 0.918 -> 0.776) outweighs the
full-range gain and 0.9933 was a val-set artifact.

**Result: 0.9960 on the shared pool, delta
+0.0113 against 0.9847, McNemar exact p = 0.001514
on 27 discordant instances
(22 won / 5 lost). The
80000-instance model BEATS the 20000-instance one.**

## Why own-validation and the shared pool disagree

`train_sudoku_model` seeds its validation instances at
`seed + n_train_instances`, so every `n_train` validates on a *different* 300
instances, and the larger training ranges cover the smaller runs' validation
seeds. At a rate near 0.99 the SE on 300 instances is ~0.005, so nothing above
~0.97 can be ranked there at all. That is what reversed the top two on 10-21
(`48912134`), and it is why the number to quote from this notebook is the
shared-pool column, never the own-val one.

In [ ]:
import csv, os, re
import numpy as np, matplotlib.pyplot as plt

S = os.environ["SCRATCH"] + "/spin_gnn"
GRID = f"{S}/sudoku_shared_pool_grid_2026-10-22"
rows = list(csv.DictReader(open(f"{GRID}/agg_paired_full_grid.csv")))
for r in rows:
    print(f"{float(r['solve_rate']):.4f}  [{float(r['wilson_lo']):.4f}, "
          f"{float(r['wilson_hi']):.4f}]  delta {float(r['delta']):+.4f}  "
          f"p {float(r['mcnemar_exact_p']):.3g}  {r['checkpoint']}")

## Metrics table

Every `shared-pool solve_rate` is the same 1500 instances; `delta`/`McNemar p` are paired against d256 x 20000 (0.9847), the previous best on this eval. `own-val agg` is the 300-instance per-run number kept only to show the divergence. Probe columns are the peak over layers, averaged over seeds 0/1/2.

| model | params | own-val agg | shared-pool solve_rate | 95% Wilson | delta vs best | McNemar p | k0 | mean_o1 | frontier AUROC | next_node |
|---|---|---|---|---|---|---|---|---|---|---|
| d128 x 5000 | -- | 0.7333 | **0.7707** | [0.7487, 0.7912] | -0.2140 | 2.11e-88 | nan | nan | 0.8577 (L5) | 0.7377 (L12) |
| d256 x 5000 | 9548834 | 0.8467 | **0.8200** | [0.7997, 0.8386] | -0.1647 | 8.38e-66 | 0.24 | 0.842 | 0.9071 (L5) | 0.7889 (L12) |
| d384 x 5000 | 16676386 | 0.8467 | **0.8760** | [0.8584, 0.8917] | -0.1087 | 2.91e-38 | 0.99 | 0.994 | 0.9160 (L10) | 0.8081 (L12) |
| d128 x 20000 | 3994146 | 0.9733 | **0.9733** | [0.9639, 0.9804] | -0.0113 | 0.0241 | 0.40 | 0.856 | 0.8831 (L12) | 0.9510 (L12) |
| d256 x 20000 | 9548834 | 0.9733 | **0.9847** | [0.9771, 0.9898] | +0.0000 | 1 | 0.60 | 0.918 | 0.9162 (L5) | 0.9640 (L12) |
| d384 x 20000 | 16676386 | 0.9867 | **0.9720** | [0.9624, 0.9792] | -0.0127 | 0.0127 | 0.91 | 0.980 | 0.9381 (L12) | 0.9566 (L12) |
| d256 x 80000 | 9548834 | 0.9933 | **0.9960** | [0.9913, 0.9982] | +0.0113 | 0.00151 | 0.07 | 0.776 | 0.9067 (L12) | 0.9895 (L12) |

## Data scaling on one eval

![](sudoku_data_scaling_2026-10-22/fig1_data_scaling.png)

## What the probes track

![](sudoku_data_scaling_2026-10-22/fig2_probes_vs_competence.png)

Over the 7 cells, Spearman against shared-pool solve_rate is **+0.96** for `next_node` peak accuracy and only **+0.25** for `frontier_mask` peak AUROC, while `frontier_mask` against `d_model` is **+0.85**.

The individual cells make the point more bluntly than the coefficient does. d256 x 5000 reaches frontier AUROC 0.907 while solving 0.820; d128 x 20000 reaches only 0.883 while solving 0.973; and the new best model, d256 x 80000, *drops* to 0.907 from the 20000-instance run's 0.916 while its solve rate rises 0.985 -> 0.996. Frontier representation strength is a function of width, and more data at fixed width can even cost a little of it.

`next_node` instead steps up with training data at every width (0.738 -> 0.951 at d128, 0.789 -> 0.964 -> 0.990 at d256, 0.810 -> 0.957 at d384) and reproduces the competence ordering with a single inversion, between d384 x 20000 and d128 x 20000 -- two models whose solve rates differ by 0.0013 and are not separable on this pool.

## The easy end goes backwards

![](sudoku_data_scaling_2026-10-22/fig3_easy_end.png)

`solve_rate_k0` is the pure-copy case (zero uncolored nodes, no reasoning required) and `mean_o1` the near-trivial end. Both are own-val metrics on 100 instances per bin, so they are noisy -- but a fall from 0.60 to 0.07 is not noise. The shape is non-monotonic: data 5000 -> 20000 *helped* the easy end (k0 0.24 -> 0.60) and 20000 -> 80000 destroyed it (0.60 -> 0.07) while the full range still rose. Width did the same thing on 10-18 (agg up, k0 collapsing), so this is not specific to one axis; it looks like the trivial cases are being traded away for the rest of the distribution rather than bought by scale of any kind.

In [ ]:
# Any pairwise McNemar, from the stored per-instance matrix -- no GPU needed.
import numpy as np, math
z = np.load(f"{GRID}/agg_paired_full_grid_per_instance.npz", allow_pickle=True)
solved, names = z["solved"], list(z["names"])

def mcnemar(i, j):
    b = int((solved[i] & ~solved[j]).sum()); c = int((~solved[i] & solved[j]).sum())
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2.0 * sum(math.comb(n, t) for t in range(k + 1)) / 2 ** n)

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        print(f"p={mcnemar(i, j):.4g}  {names[i][:34]} vs {names[j][:34]}")